In this notebook, we perform the analyses pertaining to figure 5. 
This includes 
- registration of data to the templates used in the study
- masking and translation of mouse Alzheimer's map, comparison with human map (permutation test)
- creation and translation of mouse Parkinson's map, comparison with human map (permutation test)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import os

import nibabel as nib
import nilearn
from nilearn import plotting

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import neuromaps
from neuromaps import nulls, transforms, resampling
from neuromaps import stats as nm_stats
from neuromaps.datasets import fetch_annotation

from nilearn import surface, datasets

In [ ]:
# Function definitions

def mirror_brain(half_map):
    human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
    affine = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').affine
    
    human_map = human_brain.copy()
    human_half_map = human_map[0:50,:,:]
    human_half_map[human_half_map>0] = half_map
    human_map[0:50,:,:]=human_half_map
    human_map[50:,:,:]=np.flip(human_half_map[1:49,:,:], axis=0)
    
    map_ = nib.Nifti1Image(human_map, affine)
    return map_

def translate_mouse_human(mouse_roi):
    human_labels = pd.read_csv('data/mouse_human/data.ign/atlas_ahba.csv')
    long_label = human_labels['Long label']
    
    human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
    affine = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').affine

    mouse_vox_embedding = pd.read_csv('/well/mars/users/uvy786/dl_mouse_human/exps/e3_vae/results.ign/sweep_result/encoding/mouse_voxel_encoding_with_scaling.csv')
    mouse_regions = mouse_vox_embedding.iloc[:,-1]

    similarity=np.load('similarity_scored.csv.npy')

    temp = [similarity[mouse_regions==m_roi,:] for m_roi in mouse_roi]
    translated_half_map = []
    for i in range(len(temp)):
        translated_half_map += [*temp[i]]

    translated_half_map = np.sum(translated_half_map, axis=0)
    
    scaled = (translated_half_map - translated_half_map.min()) / (translated_half_map.max() - translated_half_map.min())
    unthresholded_translated = scaled.copy()
    
    unthresholded=mirror_brain(unthresholded_translated)

    threshold=0.6 

    whole_map=unthresholded.get_fdata()
    whole_map[whole_map<threshold]=0
    whole_map = whole_map.reshape(unthresholded.get_fdata().shape)
    whole_map = nib.Nifti1Image(whole_map, unthresholded.affine)

    return whole_map, unthresholded, threshold


def plot_human_surf(mask_nii_h):
    fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage5")
    hemi= "left"
    radius = 8
    kind = 'ball'
    depth = 0.7
    inter='linear'
    n_samples=160
    pial_mesh = fsaverage[f"pial_{hemi}"]
    infl_mesh = fsaverage[f"infl_{hemi}"]
    X = surface.vol_to_surf(mask_nii_h, pial_mesh, radius=radius, interpolation=inter, n_samples=n_samples, kind=kind).T
    X = (X - X.min()) /(X.max() - X.min())
    nilearn.plotting.plot_surf_stat_map(infl_mesh,X,view=("lateral"), cmap='jet', colorbar=True)
    nilearn.plotting.plot_surf_stat_map(infl_mesh,X, view=("medial"), cmap='jet')

In [ ]:
# Data

In [ ]:
# AD

# Mouse data
minc_mouse_low = nib.load('data/mouse_human/DSURQE_CCFv3_average_200um.mnc')
affine_low = minc_mouse_low.affine
minc_mouse_high = nib.load('data/mouse_human/DSURQE_CCFv3_average_50um.mnc')
affine_high=minc_mouse_high.affine
analysis_mask = pd.read_csv('data/mouse_human/mouse_analysis_mask.csv')
analysis_mask = analysis_mask['x']

AD_map = nib.load('data/mouse_human/AD/resliced_AD_map_AAP_PSI_13mo.nii.gz')
AD_data = AD_map.get_fdata()
affine_flip = AD_map.affine*[[-1,0,0,-1],[0,-1,0,-1],[0,0,1,1],[0,0,0,1]]
AD_flip_map=nib.Nifti1Image(AD_data,affine_flip)

AD_data = AD_flip_map.get_fdata()

# AD_data=(AD_data - AD_data.min())/(AD_data.max() - AD_data.min())

AD_data.flatten()[analysis_mask==False]=0
AD_data.reshape(minc_mouse_low.shape)

nilearn.plotting.view_img(AD_flip_map, bg_img=minc_mouse_low, cmap='hot', draw_cross=False, symmetric_cmap=False,vmin=0)


In [ ]:
# Translate mouse
# Wholebrain
sim=np.load('similarity_scored.csv.npy') # OR: compute similarity (see tutorial)

trans_AD = np.dot(AD_data.flatten()[analysis_mask],sim)
std = StandardScaler()
trans_AD = std.fit_transform(trans_AD.reshape(-1,1))
map_nii = mirror_brain(np.squeeze(trans_AD))
nilearn.plotting.view_img(map_nii, vmin=0, symmetric_cmap=False)

# nib.save(map_nii, 'translated_vox_AD.nii.gz') 


In [ ]:
# Human data

AD_real_data=nib.load('data/mouse_human/AD/0004_T_NC_vs_AD.nii')

AD_real_registered = transforms.mni152_to_mni152('data/mouse_human/AD/0004_T_NC_vs_AD.nii',target='translated_vox_AD.nii.gz',method='linear')


human_brain = nib.load('data/mouse_human/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
human_brain=human_brain[0:50,:,:].flatten()

AD_real_registered_data=AD_real_registered.get_fdata()
AD_real_registered_data[np.isnan(AD_real_registered_data)]=0


AD_real_registered_data = AD_real_registered_data[0:50,:,:].flatten()
AD_real_registered_data=AD_real_registered_data[human_brain>0]



AD = AD_real_registered_data
AD = StandardScaler().fit_transform(AD.reshape(-1,1))
# AD = (AD - AD.min())/(AD.max()-AD.min())
AD = AD.reshape(AD_real_registered_data.shape)

AD_real_full=mirror_brain(AD)

nilearn.plotting.view_img(AD_real_full)

In [ ]:
# Compare translation and real human data 
data1 = AD
data2 = trans_AD

mask = np.zeros(len(data1.flatten()))
mask[data1.flatten()>0]=1
data1 = data1.flatten()[mask>0]
data2 = data2.flatten()[mask>0]

corr=neuromaps.stats.efficient_pearsonr(data1, data2, nan_policy='propagate', return_pval=False)




# Compute pval using permutations
from sklearn.utils.validation import check_random_state
from scipy.stats._stats_py import _chk2_asarray

true_sim = corr
# abs_true = np.abs(true_sim)
n_perm=1000
data1, data2, _ = _chk2_asarray(data1, data2, 0)
rs = check_random_state(0)
permutations = np.ones(true_sim.shape)
nulldist = np.zeros(((n_perm, ) + true_sim.shape))
for perm in range(n_perm):
    # permute `a` and determine whether correlations exceed original
    ap = data1[rs.permutation(len(data1))] 
    nullcomp = neuromaps.stats.efficient_pearsonr(ap, data2, nan_policy='omit',return_pval=False)
    permutations += np.abs(nullcomp) >= corr
    nulldist[perm] = nullcomp

pvals = permutations / (n_perm + 1)

In [ ]:
corr, pvals

In [ ]:
from nilearn.image import resample_img
from nilearn.decoding import SearchLight
from sklearn.metrics import make_scorer, r2_score

from scipy.stats import pearsonr

def pearson_scorer(true_vals, preds):
    """Custom scoring function that converts to Pearson's r."""
    r, _ = pearsonr(true_vals, preds)
    return r

pearson_score = make_scorer(pearson_scorer, greater_is_better=True)

import nibabel as nib

pred_4d = nib.Nifti1Image(map_nii.get_fdata()[..., np.newaxis],
                                affine=map_nii.affine)

true_4d = nib.Nifti1Image(AD_real_full.get_fdata()[..., np.newaxis],
                                affine=AD_real_full.affine)

X = pred_4d  # Our "samples" 
y = AD_real_full.get_fdata()[..., 0]  # Ground truth for scoring

from nilearn.masking import compute_brain_mask

mask_img = compute_brain_mask(pred_4d)

searchlight = SearchLight(mask_img=None,  # or your mask
                           process_mask_img=mask_img,
                           radius=3,  # in voxel
                           scoring=pearson_score,
                           n_jobs=-1,  # use all cores
                           verbose=1)

searchlight.fit([X], y)  # fitting on single sample
searchlight_img = searchlight.scores_

from nilearn.plotting import view_img, plot_stat_map

plot_stat_map(searchlight_img, title='Searchlight Pearson')

In [ ]:
# Translate cortex only
# Get a mask of the cortex in MNI 
human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
human_brain=human_brain[0:50,:,:].flatten()
human_brain=human_brain[human_brain>0]
cortical_brain_mask=np.zeros(human_brain.shape)

rois = [np.arange(1,6),np.arange(51,90),[90,91,92,93,94,95,106],np.arange(108,113)] # with/without hippocampus [90,91,92]
rois_list = []
for i in range(len(rois)):
    rois_list += [*rois[i]]

for roi in range(len(rois_list)):
    cortical_brain_mask = cortical_brain_mask + 1*(human_brain==rois_list[roi])

# Get cortical vector mapping similarity matrix
cortex_vec=np.zeros(67)
cortex_vec[[14,15,16,18,19, 20, 21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36]] = 1 # with / without hippocampus (14-15-16)
mouse_labels = pd.read_csv("data/mouse_human/reordered_mouse.csv")
mouse_regions_cx = mouse_labels['Region'][cortex_vec==1].reset_index(drop=True)
data_mouse=pd.read_csv('data/mouse_human/genes/MouseExpressionMatrix_voxel_coronal_maskcoronal_log2_grouped_imputed_labelled_scaled.csv')['Region67']
temp=[]
for i in range(len(mouse_regions_cx)):
    temp.append(np.where(data_mouse==mouse_regions_cx[i]))

def flatten(xss):
    return [x for xs in xss for x in xs]

inds_cx_mouse = flatten(flatten(temp))

cortex_vox_vec = np.zeros(len(data_mouse))

del data_mouse
del mouse_labels

cortex_vox_vec[inds_cx_mouse]=1


In [ ]:
# Get cortical sim

cortical_sim=sim[:,cortical_brain_mask>0]

del sim #Free up ram

cortical_sim_ = cortical_sim[cortex_vox_vec==1,:]

del cortical_sim

In [ ]:
cortex_vox_vec[inds_cx_mouse]=AD_data.flatten()[inds_cx_mouse]

translated_cortical_AD_map = np.dot(cortical_sim_.T,cortex_vox_vec[inds_cx_mouse])

translated_cortical_AD_map = std.fit_transform(translated_cortical_AD_map.reshape(-1,1))

translated_full_cortical_AD_map=cortical_brain_mask.copy()
translated_full_cortical_AD_map[cortical_brain_mask==1]=np.squeeze(translated_cortical_AD_map)
map_nii = mirror_brain(translated_full_cortical_AD_map)
nilearn.plotting.view_img(map_nii, vmin=0, symmetric_cmap=False)

In [ ]:
# Compare with cortical AD
real_cx_AD = cortical_brain_mask.copy()
real_cx_AD[cortical_brain_mask==1]=AD[cortical_brain_mask==1]
real_cx_nii = mirror_brain(real_cx_AD)

In [ ]:
# Stats on the surface using Neuromaps

# First: put data into fsaverage space 
pathnii = 'translated_vox_AD.nii.gz'

pathnii_2 = 'real_cx_AD.nii.gz'

affine = nib.load(pathnii).affine
dat1 = nib.load(pathnii).get_fdata()
dat1 = (dat1 - dat1.min()) / (dat1.max() - dat1.min())
nib.save(nib.Nifti1Image(dat1,affine), 'translated_vox_AD_scaled.nii.gz')

dat2 = nib.load(pathnii_2).get_fdata()
dat2 = (dat2 - dat2.min()) / (dat2.max() - dat2.min())
nib.save(nib.Nifti1Image(dat2,affine), 'real_cx_AD_scaled.nii.gz')


gii_files = ['fsaverage_translated_AD_scaled_L.gii','fsaverage_translated_AD_scaled_R.gii']
gii_files_2 = ['fsaverage_AD_real_scaled_L.gii', 'fsaverage_AD_real_scaled_R.gii']

fsaverage_h_pc = transforms.mni152_to_fsaverage(pathnii_scaled, '10k')
fsaverage_2 = transforms.mni152_to_fsaverage(pathnii_2_scaled, '10k')
nib.save(fsaverage_h_pc[0], gii_files[0])
nib.save(fsaverage_h_pc[1], gii_files[1])
nib.save(fsaverage_2[0], gii_files_2[0])
nib.save(fsaverage_2[1], gii_files_2[1])

# spin test 

rotated = nulls.alexander_bloch(gii_files_2, atlas='fsaverage', density='10k', n_perm=1000, seed=123)

corr, pval = nm_stats.compare_images(gii_files_2, gii_files, nulls=rotated)
print(corr)
print(pval)

In [ ]:
# Subcortex only


In [ ]:
# PD
# NB: mouse masks based on the MPTP mouse model effects

# Real data:
# Coregister the real AD map to translated map and get subcortical data only

PD_real_data=nib.load('data/mouse_human/PD/dr_stage3_ic0000_tstat2_PDvsHC.nii.gz')

PD_real_registered = transforms.mni152_to_mni152('data/mouse_human/PD/dr_stage3_ic0000_tstat2_PDvsHC.nii.gz',target='translated_vox_AD.nii.gz',method='linear')

human_brain = nib.load('data/mouse_human/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
human_brain=human_brain[0:50,:,:].flatten()

PD_real_registered_data=PD_real_registered.get_fdata()
PD_real_registered_data = PD_real_registered_data[0:50,:,:].flatten()
PD_real_registered_data=PD_real_registered_data[human_brain>0]

# #  make subcx brain mask
human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
human_brain=human_brain[0:50,:,:].flatten()
human_brain=human_brain[human_brain>0]
subcx_brain_mask=np.zeros(human_brain.shape)

rois_subcx = [np.arange(6,51), [90,91,92,99,104,107]]
rois_subcx_list = []
for i in range(len(rois_subcx)):
    rois_subcx_list += [*rois_subcx[i]]
    
for roi in range(len(rois_subcx_list)):
    subcx_brain_mask =subcx_brain_mask + 1*(human_brain==rois_subcx_list[roi])


PD_sbcx = PD_real_registered_data.copy()
PD_sbcx[subcx_brain_mask==0]=np.nan
scaler = MinMaxScaler()
PD_sbcx = np.squeeze(scaler.fit_transform(PD_real_registered_data.reshape(-1,1)))
PD_sbcx[subcx_brain_mask==0]=0
map_nii=mirror_brain(PD_sbcx)
nilearn.plotting.view_img(map_nii)
# nib.save(map_nii, 'real_subcx_PD_map_scaled.nii')

# nilearn.plotting.plot_stat_map(map_nii, draw_cross=False)

In [ ]:
# Generate mouse map from article: 

PD_vec=np.zeros(67)
PD_vec[[37,40,43,45,47]] = 1  # STN (51) should have -1, but since most areas of the hypothalamus have +1 this is the value that will be retained for area 51.
PD_vec[[0,3,38,39,41,44,49,50]] = -1 # BST should have +1 but we kept GPi for area 39

minc_labels = nib.load('data/mouse_human/Atlas_67.mnc')
affine_labels = minc_labels.affine

analysis_mask = pd.read_csv('data/mouse_human/mouse_analysis_mask.csv')
analysis_mask = analysis_mask['x']

mouse_mask = np.rint(minc_labels.get_fdata()).flatten()
for i in range(67):
    mouse_mask[mouse_mask==i+1]=PD_vec[i]

mouse_mask = mouse_mask.reshape(minc_labels.get_fdata().shape)
mask_nii_h = nib.Nifti1Image(mouse_mask, affine=affine_labels)



In [ ]:
# Make subcortical similarity matrix

human_brain = nib.load('data/mouse_human/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
human_brain=human_brain[0:50,:,:].flatten()
human_brain=human_brain[human_brain>0]
subcx_brain_mask=np.zeros(human_brain.shape)

rois_subcx = [np.arange(6,51), [90,91,92,99,104,107]]
rois_subcx_list = []
for i in range(len(rois_subcx)):
    rois_subcx_list += [*rois_subcx[i]]
    
for roi in range(len(rois_subcx_list)):
    subcx_brain_mask =subcx_brain_mask + 1*(human_brain==rois_subcx_list[roi])

# Human indices
sim = np.load('similarity_scored.csv.npy')
subcx_sim = sim[:,subcx_brain_mask==1]

del sim

In [ ]:
# Mouse indices
# NB. mouse_human/genes/part_* need to be put back together before use as MouseExpressionMatrix

subcortex_vec = np.zeros(67)
subcortex_vec[[0,6,7,14,15,16,37,38,39,40,41, 49,50]] = 1

mouse_labels = pd.read_csv("data/mouse_human/reordered_mouse.csv")
mouse_regions_subcx = mouse_labels['Region'][subcortex_vec==1].reset_index(drop=True)
data_mouse=pd.read_csv('data/mouse_human/genes/MouseExpressionMatrix_voxel_coronal_maskcoronal_log2_grouped_imputed_labelled_scaled.csv')['Region67']
temp=[]
for i in range(len(mouse_regions_subcx)):
    temp.append(np.where(data_mouse==mouse_regions_subcx[i]))

def flatten(xss):
    return [x for xs in xss for x in xs]

inds_subcx_mouse = flatten(flatten(temp))
subcx_vox_vec = np.zeros(len(data_mouse))
del data_mouse
del mouse_labels
subcx_vox_vec[inds_subcx_mouse]=1
subcx_sim = subcx_sim[subcx_vox_vec==1,:]

In [ ]:

# #  Subcortex
subcx_vox_PD = np.zeros(subcx_vox_vec.shape)
subcx_vox_PD[inds_subcx_mouse]=mask_nii_h.get_fdata().flatten()[inds_subcx_mouse]


translated_subcx_PD_map = np.dot(subcx_sim.T,subcx_vox_PD[inds_subcx_mouse])

std = StandardScaler()
translated_PD = np.squeeze(std.fit_transform(translated_subcx_PD_map.reshape(-1,1)))

In [ ]:
translated_full_sbcx_PD_map=subcx_brain_mask.copy()
translated_full_sbcx_PD_map[subcx_brain_mask==1]=translated_PD
translated_full_sbcx_PD_map[subcx_brain_mask==0]=0
map_nii = mirror_brain(translated_full_sbcx_PD_map)
nilearn.plotting.view_img(map_nii)

In [ ]:
# Stats
data1 = PD_sbcx
data1=data1[subcx_brain_mask>0]
data2 = translated_full_sbcx_PD_map
data2=data2[subcx_brain_mask>0]

mask = np.zeros(len(data1.flatten()))
mask[data1.flatten()>0]=1

data1 = data1.flatten()
data2 = data2.flatten()
corr=nm_stats.compare_images(data1, data2)

In [ ]:
# Do permutation test
true_sim = corr
n_perm=1000
rs = check_random_state(0)
permutations = np.ones(true_sim.shape)
nulldist = np.zeros(((n_perm, ) + true_sim.shape))
for perm in range(n_perm):
    # permute `a` and determine whether correlations exceed original
    ap = data1[rs.permutation(len(data1))] 
    nullcomp = neuromaps.stats.efficient_pearsonr(ap, data2, nan_policy='omit',return_pval=False)
    permutations += np.abs(nullcomp) >= corr
    nulldist[perm] = nullcomp

permutations / (n_perm + 1) 